Name :  Bhalchim Roshani Ramchandra  
Roll No.: 08  
Title : Data Wrangling  
Problem Statement: Data Wrangling on Real Estate Market  
Dataset: "RealEstate_Prices.csv"  
Description: The dataset contains information about housing prices in a specific real estate
market. It includes various attributes such as property characteristics, location, sale prices,
and other relevant features. The goal is to perform data wrangling to gain insights into the
factors influencing housing prices and prepare the dataset for further analysis or modeling.
Tasks to Perform:
1. Import the "RealEstate_Prices.csv" dataset. Clean column names by removing spaces,
special characters, or renaming them for clarity.
2. Handle missing values in the dataset, deciding on an appropriate strategy (e.g.,
imputation or removal).
3. Perform data merging if additional datasets with relevant information are available
(e.g., neighborhood demographics or nearby amenities).
4. Filter and subset the data based on specific criteria, such as a particular time period,
property type, or location.
5. Handle categorical variables by encoding them appropriately (e.g., one-hot encoding or
label encoding) for further analysis.
6. Aggregate the data to calculate summary statistics or derived metrics such as average
sale prices by neighborhood or property type.
7. Identify and handle outliers or extreme values in the data that may affect the analysis
or modeling process.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('RealEstate_Prices.csv')
print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

Dataset loaded successfully!
Dataset Shape: (2051, 81)


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-zA-Z0-9]+', '_', regex=True)
    .str.strip('_')
)

print("Column names cleaned successfully!")

Column names cleaned successfully!


In [4]:
print("Missing Values in Each Column:")
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

print("\nTotal Missing Values:", df.isnull().sum().sum())

Missing Values in Each Column:
pool_qc           2042
misc_feature      1986
alley             1911
fence             1651
mas_vnr_type      1240
fireplace_qu      1000
lot_frontage       330
garage_cond        114
garage_yr_blt      114
garage_finish      114
garage_qual        114
garage_type        113
bsmt_exposure       58
bsmtfin_type_2      56
bsmt_qual           55
bsmtfin_type_1      55
bsmt_cond           55
mas_vnr_area        22
bsmt_half_bath       2
bsmt_full_bath       2
bsmtfin_sf_1         1
total_bsmt_sf        1
bsmt_unf_sf          1
bsmtfin_sf_2         1
garage_area          1
garage_cars          1
dtype: int64

Total Missing Values: 11040


In [5]:

for col in df.select_dtypes(include='object').columns:
    if col in ['alley', 'pool_qc', 'misc_feature', 'fence',
               'mas_vnr_type', 'fireplace_qu', 'garage_type',
               'garage_finish', 'garage_qual', 'garage_cond',
               'bsmt_qual', 'bsmt_cond', 'bsmt_exposure',
               'bsmtfin_type_1', 'bsmtfin_type_2']:
        df[col] = df[col].fillna('None')
    else:
        df[col] = df[col].fillna('Unknown')


for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

print("Missing values handled successfully!")
print("Total Missing Values Remaining:", df.isnull().sum().sum())

Missing values handled successfully!
Total Missing Values Remaining: 0


In [6]:
# Create a small neighborhood reference dataset
neighborhood_data = {
    'neighborhood': df['neighborhood'].unique(),
    'area_type': ['Residential'] * df['neighborhood'].nunique()
}

df_neighborhood = pd.DataFrame(neighborhood_data)

# Merge the datasets
df = pd.merge(df, df_neighborhood, on='neighborhood', how='left')

print("Datasets merged successfully!")
print("Merged Dataset Shape:", df.shape)

display(df[['neighborhood', 'area_type', 'saleprice']].head())


Datasets merged successfully!
Merged Dataset Shape: (2051, 82)


,neighborhood,area_type,saleprice
0,Sawyer,Residential,130500
1,SawyerW,Residential,220000
2,NAmes,Residential,109000
3,Timber,Residential,174000
4,SawyerW,Residential,138500


In [7]:
# Filter houses sold in 2010 with sale price above $200,000
filtered_df = df[
    (df['yr_sold'] == 2010) &
    (df['saleprice'] > 200000)
]

print("Filtered Dataset Shape:", filtered_df.shape)

print("\nFiltered House Records:")
display(filtered_df[['neighborhood', 'yr_sold', 'saleprice']].head(10))

Filtered Dataset Shape: (67, 82)

Filtered House Records:


,neighborhood,yr_sold,saleprice
19,Somerst,2010,215200
60,NoRidge,2010,290000
62,ClearCr,2010,270000
70,Edwards,2010,308030
144,Somerst,2010,267916
172,Somerst,2010,204500
253,CollgCr,2010,272000
316,NridgHt,2010,319900
369,Timber,2010,269500
398,NridgHt,2010,376162


In [8]:
# Select categorical columns for encoding
categorical_cols = ['neighborhood', 'bldg_type', 'house_style']

# Apply one-hot encoding
df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    dtype=int
)

print("Categorical variables encoded successfully!")
print("Original Dataset Shape:", df.shape)
print("Encoded Dataset Shape:", df_encoded.shape)

display(df_encoded.head())

Categorical variables encoded successfully!
Original Dataset Shape: (2051, 82)
Encoded Dataset Shape: (2051, 120)


,id,pid,ms_subclass,ms_zoning,lot_frontage,lot_area,street,alley,lot_shape,land_contour,...,bldg_type_Twnhs,bldg_type_TwnhsE,house_style_1.5Fin,house_style_1.5Unf,house_style_1Story,house_style_2.5Fin,house_style_2.5Unf,house_style_2Story,house_style_SFoyer,house_style_SLvl
0,109,533352170,60,RL,68.0,13517,Pave,None,IR1,Lvl,...,0,0,0,0,0,0,0,1,0,0
1,544,531379050,60,RL,43.0,11492,Pave,None,IR1,Lvl,...,0,0,0,0,0,0,0,1,0,0
2,153,535304180,20,RL,68.0,7922,Pave,None,Reg,Lvl,...,0,0,0,0,1,0,0,0,0,0
3,318,916386060,60,RL,73.0,9802,Pave,None,Reg,Lvl,...,0,0,0,0,0,0,0,1,0,0
4,255,906425045,50,RL,82.0,14235,Pave,None,IR1,Lvl,...,0,0,1,0,0,0,0,0,0,0


In [9]:
# Calculate average sale price for each neighborhood
neighborhood_prices = df.groupby('neighborhood')['saleprice'].mean()

# Sort neighborhoods by average sale price
neighborhood_prices = neighborhood_prices.sort_values(ascending=False)

print("Average Sale Price by Neighborhood:")
display(neighborhood_prices.head(10))

Average Sale Price by Neighborhood:


neighborhood
StoneBr    329675.736842
NridgHt    322831.352459
NoRidge    316294.125000
GrnHill    280000.000000
Veenker    253570.588235
Timber     241051.354167
Somerst    227183.900000
ClearCr    217490.074074
Crawfor    205901.211268
CollgCr    202497.216667
Name: saleprice, dtype: float64

In [10]:
for col in ['saleprice', 'gr_liv_area']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower_limit) | (df[col] > upper_limit)).sum()

    print(f"{col}: {outliers} outliers detected")

    # Cap outliers within the IQR limits
    df[col] = df[col].clip(lower=lower_limit, upper=upper_limit)

print("\nOutlier handling completed!")
print("Final Dataset Shape:", df.shape)

saleprice: 94 outliers detected
gr_liv_area: 64 outliers detected

Outlier handling completed!
Final Dataset Shape: (2051, 82)


In [11]:
df.to_csv('RealEstate_Prices_Wrangled.csv', index=False)

print("Final dataset exported successfully!")
print("File Name: RealEstate_Prices_Wrangled.csv")
print("Final Dataset Shape:", df.shape)

Final dataset exported successfully!
File Name: RealEstate_Prices_Wrangled.csv
Final Dataset Shape: (2051, 82)
